# hydra_pipeline

> Shared Hydra config helpers for `scripts/build.py` and `scripts/train.py`.

Channel presets, patch-cache orchestration, and stats loading — kept in the
library (not under `scripts/`) so `segpt-build` / `segpt-train` can import it
after `pip install` or `uv sync`.

In [ ]:
#| default_exp hydra_pipeline

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import json
from pathlib import Path

import cv2
from omegaconf import DictConfig

from segmentation_tools_pt.channels import CHANNEL_OPS, build_image, tophat
from segmentation_tools_pt.dataset import split_sample_ids, build_direct_dataloaders
from segmentation_tools_pt.dataset_build import build_patch_dataset
from segmentation_tools_pt.patches import PatchConfig
from segmentation_tools_pt.transforms import boundary_blur_kwargs_from_cfg, compute_dataset_stats

__all__ = [
    'CHANNEL_PRESETS',
    'resolve_channel_preset',
    'resolve_run_seed',
    'resolve_deterministic',
    'resolve_task',
    'resolve_mask_cfg',
    'validate_mask_mode',
    'resolve_hierarchy_pairs',
    'resolve_hierarchy_weight',
    'validate_patch_training_sizes',
    'patches_exist',
    'iter_train_images',
    'iter_raw_train_images',
    'load_or_compute_stats',
    'load_or_compute_stats_direct',
    'run_patch_pipeline',
    'run_direct_pipeline',
    'validate_data_mode_channels',
    'resolve_boundary_blur_cfg',
    'resolve_batch_aug_cfg',
]

CHANNEL_OPS['ch2'] = lambda imgs: imgs[2]
CHANNEL_OPS['tophat_ch0'] = lambda imgs: tophat(imgs[0], 21)
CHANNEL_OPS['sobel_ch0'] = lambda imgs: cv2.convertScaleAbs(
    cv2.Sobel(imgs[0], cv2.CV_32F, 1, 1, ksize=3))

CHANNEL_PRESETS = {
    'single': dict(n_source_channels=1, channel_names=['ch0'], in_channels=1),
    'single_repeat_3': dict(
        n_source_channels=1, channel_names=['ch0', 'ch0', 'ch0'], in_channels=3),
    'single_to_3': dict(
        n_source_channels=1,
        channel_names=['ch0', 'tophat_ch0', 'sobel_ch0'],
        in_channels=3),
    'two_channel': dict(
        n_source_channels=2, channel_names=['ch0', 'ch1', 'product'], in_channels=3),
    'three_channel': dict(
        n_source_channels=3, channel_names=['ch0', 'ch1', 'ch2'], in_channels=3),
}

In [ ]:
#| export
def resolve_channel_preset(data_mode: str) -> dict:
    """Map `data.data_mode` to source channel count, names, and model `in_channels`.

    Example:
        >>> resolve_channel_preset('single')['in_channels']
        1
    """
    if data_mode not in CHANNEL_PRESETS:
        raise ValueError(
            f'Unknown data.data_mode={data_mode!r}. '
            f'Choose one of: {list(CHANNEL_PRESETS)}'
        )
    return CHANNEL_PRESETS[data_mode]


def resolve_run_seed(cfg: DictConfig) -> int:
    """Return the run seed, preferring `train.seed` and falling back to `patch.seed`."""
    return int(cfg.train.get('seed', cfg.patch.seed))


def resolve_deterministic(cfg: DictConfig) -> bool:
    """Return whether the run should request deterministic torch backends."""
    return bool(cfg.train.get('deterministic', True))


def resolve_task(cfg: DictConfig):
    """Return `train.task` ('binary'|'multiclass'|'multilabel'), or None if unset (auto-infer)."""
    return cfg.train.get('task', None)


def resolve_mask_cfg(cfg: DictConfig) -> dict:
    """Return `{mask_mode, mask_channels, class_names}` from `data.*`, defaulting to 'single'."""
    class_names = cfg.data.get('class_names', None)
    if class_names is not None:
        class_names = list(class_names)
    return dict(
        mask_mode=cfg.data.get('mask_mode', 'single'),
        mask_channels=cfg.data.get('mask_channels', None),
        class_names=class_names,
    )


def validate_mask_mode(cfg: DictConfig) -> tuple[str, int | None, list[str] | None]:
    """Resolve `data.mask_mode` and validate its companion fields.

    Returns `(mask_mode, mask_channels, class_names)`, mirroring `validate_data_mode_channels`'s
    tuple-unpacking shape. `mask_mode='stacked'` requires `data.mask_channels`;
    `mask_mode='per_class_files'` requires `data.class_names`.

    Example:
        >>> validate_mask_mode(OmegaConf.create({'data': {}}))
        ('single', None, None)
    """
    mask_cfg = resolve_mask_cfg(cfg)
    mask_mode = mask_cfg['mask_mode']
    if mask_mode not in ('single', 'stacked', 'per_class_files'):
        raise ValueError(
            f'Unknown data.mask_mode={mask_mode!r}. '
            "Choose one of: ['single', 'stacked', 'per_class_files']"
        )
    if mask_mode == 'stacked' and mask_cfg['mask_channels'] is None:
        raise ValueError(
            "data.mask_mode='stacked' requires data.mask_channels (int number of stacked mask channels)"
        )
    if mask_mode == 'per_class_files' and not mask_cfg['class_names']:
        raise ValueError(
            "data.mask_mode='per_class_files' requires data.class_names (per-class mask filename suffixes)"
        )
    return mask_mode, mask_cfg['mask_channels'], mask_cfg['class_names']


def resolve_hierarchy_pairs(cfg: DictConfig, class_names: list[str] | None) -> list[tuple[int, int]]:
    """Parse `train.hierarchy.<sub>_subset_of_<super>: true` entries into `(sub_idx, super_idx)` pairs.

    E.g. `train.hierarchy.lifted_subset_of_chip: true` with `class_names=['chip','void','lifted']`
    -> `[(2, 0)]`, feeding directly into `SegLoss(hierarchy_pairs=...)` and `apply_hierarchy()`.
    Returns `[]` when there's no hierarchy section or no `class_names` to resolve indices against.
    """
    hierarchy = cfg.train.get('hierarchy', None)
    if not hierarchy or class_names is None:
        return []
    pairs = []
    for key, enabled in hierarchy.items():
        if not enabled or '_subset_of_' not in key:
            continue
        sub, _, sup = key.partition('_subset_of_')
        if sub in class_names and sup in class_names:
            pairs.append((class_names.index(sub), class_names.index(sup)))
    return pairs


def validate_patch_training_sizes(cfg: DictConfig) -> None:
    """Fail early when the model crop is larger than the saved patch size."""
    patch_size = int(cfg.patch.patch_size)
    crop_size = int(cfg.train.crop_size)
    if crop_size > patch_size:
        raise ValueError(
            'Invalid patch-mode configuration: '
            f'train.crop_size={crop_size} is larger than patch.patch_size={patch_size}. '
            'In patch mode the model crop must fit inside the saved patch. '
            'Use a larger patch.patch_size, or reduce train.crop_size.'
        )

## Class-targeted morph config (`postprocess.class_morph`)

`inference.apply_class_morph` is inference-only and opt-in: a YAML block with a
master switch default **false**. `resolve_class_morph_cfg` maps the block to the
integer arguments `apply_class_morph` needs, resolving `class_name` against
`data.class_names`. Wrong configs (enabled without a name, unknown name, no
class_names) must fail loudly — never silently clean the wrong channel.

In [ ]:
# experiment: resolve 'void' against class_names by hand to see the shape of the answer
from omegaconf import OmegaConf

cfg_x = OmegaConf.create({
    'postprocess': {'class_morph': {'enabled': True, 'class_name': 'void'}},
    'data': {'class_names': ['chip', 'void', 'lifted']},
})
names_x = list(cfg_x.data.class_names)
name_x = cfg_x.postprocess.class_morph.class_name
idx_x = names_x.index(name_x)
print(f'{name_x!r} -> class_idx={idx_x}')  # expect 1
assert idx_x == 1

In [ ]:
#| export
def resolve_class_morph_cfg(cfg: DictConfig, class_names: list[str] | None) -> dict | None:
    """Map Hydra `postprocess.class_morph.*` to `apply_class_morph(...)` args; None = disabled (no-op).

    Missing block or `enabled: false` -> None (decode unchanged). When enabled,
    `class_name` must exist in `class_names`; anything else raises a clear
    ValueError. Inference/production decode only — never loss or raw train metrics.

    Example:
        >>> resolve_class_morph_cfg(OmegaConf.create({'postprocess': {'class_morph':
        ...     {'enabled': True, 'class_name': 'void'}}, 'data': {'class_names': ['chip', 'void', 'lifted']}}))
        {'class_idx': 1, 'kernel_size': 3, 'min_area': 16}
    """
    post = cfg.get('postprocess', None)
    cm = post.get('class_morph', None) if post is not None else None
    if cm is None or not bool(cm.get('enabled', False)):
        return None
    if class_names is None:
        raise ValueError(
            'postprocess.class_morph.enabled=true requires data.class_names '
            'to resolve postprocess.class_morph.class_name'
        )
    name = cm.get('class_name', None)
    if name is None:
        raise ValueError(
            'postprocess.class_morph.enabled=true requires postprocess.class_morph.class_name'
        )
    if name not in class_names:
        raise ValueError(
            f'Unknown postprocess.class_morph.class_name={name!r}. '
            f'Choose one of: {list(class_names)}'
        )
    kernel_size = int(cm.get('kernel_size', 3))
    min_area = int(cm.get('min_area', 16))
    if kernel_size < 1:
        raise ValueError(f'postprocess.class_morph.kernel_size must be >= 1, got {kernel_size}')
    if min_area < 1:
        raise ValueError(f'postprocess.class_morph.min_area must be >= 1, got {min_area}')
    return {'class_idx': class_names.index(name), 'kernel_size': kernel_size, 'min_area': min_area}

In [ ]:
#| test
# test: class_morph resolver — off/missing = None (strict no-op); enabled resolves; bad config fails clearly
names = ['chip', 'void', 'lifted']

assert resolve_class_morph_cfg(OmegaConf.create({'postprocess': {'class_morph': {'enabled': False, 'class_name': 'void'}}}), names) is None
assert resolve_class_morph_cfg(OmegaConf.create({'data': {'class_names': names}}), names) is None  # no block -> old configs keep working

on = OmegaConf.create({'postprocess': {'class_morph': {'enabled': True, 'class_name': 'void', 'kernel_size': 5, 'min_area': 32}}})
assert resolve_class_morph_cfg(on, names) == {'class_idx': 1, 'kernel_size': 5, 'min_area': 32}

dflt = OmegaConf.create({'postprocess': {'class_morph': {'enabled': True, 'class_name': 'chip'}}})
assert resolve_class_morph_cfg(dflt, names) == {'class_idx': 0, 'kernel_size': 3, 'min_area': 16}  # defaults

def _raises(fn, msg):
    try:
        fn()
        raise AssertionError(f'expected ValueError: {msg}')
    except ValueError as e:
        assert msg in str(e), str(e)

_no_name = OmegaConf.create({'postprocess': {'class_morph': {'enabled': True}}})
_raises(lambda: resolve_class_morph_cfg(_no_name, names), 'requires postprocess.class_morph.class_name')
_no_names = OmegaConf.create({'postprocess': {'class_morph': {'enabled': True, 'class_name': 'void'}}})
_raises(lambda: resolve_class_morph_cfg(_no_names, None), 'requires data.class_names')
_bad = OmegaConf.create({'postprocess': {'class_morph': {'enabled': True, 'class_name': 'crack'}}})
_raises(lambda: resolve_class_morph_cfg(_bad, names), "Unknown postprocess.class_morph.class_name='crack'")
print('resolve_class_morph_cfg ✓  off=None, name→idx, defaults, clear errors on missing/unknown class')

In [ ]:
# test
assert resolve_channel_preset('single_repeat_3')['in_channels'] == 3

In [ ]:
# test: resolve_task / resolve_mask_cfg read train.task / data.mask_mode, defaulting sanely
from omegaconf import OmegaConf

cfg_default = OmegaConf.create({'train': {'num_classes': 1}, 'data': {}})
assert resolve_task(cfg_default) is None
mc = resolve_mask_cfg(cfg_default)
assert mc == {'mask_mode': 'single', 'mask_channels': None, 'class_names': None}
assert resolve_hierarchy_pairs(cfg_default, mc['class_names']) == []

cfg_ml = OmegaConf.create({
    'train': {'num_classes': 3, 'task': 'multilabel', 'hierarchy': {'lifted_subset_of_chip': True}},
    'data': {'mask_mode': 'per_class_files', 'class_names': ['chip', 'void', 'lifted']},
})
assert resolve_task(cfg_ml) == 'multilabel'
mc_ml = resolve_mask_cfg(cfg_ml)
assert mc_ml == {'mask_mode': 'per_class_files', 'mask_channels': None, 'class_names': ['chip', 'void', 'lifted']}
assert resolve_hierarchy_pairs(cfg_ml, mc_ml['class_names']) == [(2, 0)]
print('resolve_task / resolve_mask_cfg / resolve_hierarchy_pairs ✓  default single-task unchanged, multilabel keys and hierarchy pairs read correctly')

In [ ]:
# test: validate_mask_mode validates data.mask_mode and its companion fields
assert validate_mask_mode(cfg_default) == ('single', None, None)
assert validate_mask_mode(cfg_ml) == ('per_class_files', None, ['chip', 'void', 'lifted'])

cfg_stacked = OmegaConf.create({'train': {'num_classes': 3}, 'data': {'mask_mode': 'stacked', 'mask_channels': 3}})
assert validate_mask_mode(cfg_stacked) == ('stacked', 3, None)

try:
    validate_mask_mode(OmegaConf.create({'train': {}, 'data': {'mask_mode': 'bogus'}}))
    assert False, 'expected ValueError for unknown mask_mode'
except ValueError:
    pass

try:
    validate_mask_mode(OmegaConf.create({'train': {}, 'data': {'mask_mode': 'stacked'}}))
    assert False, 'expected ValueError for stacked without mask_channels'
except ValueError:
    pass

try:
    validate_mask_mode(OmegaConf.create({'train': {}, 'data': {'mask_mode': 'per_class_files'}}))
    assert False, 'expected ValueError for per_class_files without class_names'
except ValueError:
    pass

print('validate_mask_mode ✓  valid modes pass through, invalid/missing companion fields raise')

In [ ]:
#| export
def resolve_boundary_blur_cfg(cfg: DictConfig) -> dict:
    """Map Hydra `augment.boundary_*` keys to `get_train_transforms(boundary_blur=...)`."""
    return boundary_blur_kwargs_from_cfg(cfg.augment)


In [ ]:
#| export
def patches_exist(patch_dir: Path) -> bool:
    """Return True when `patch_dir/metadata.json` lists train patches."""
    meta = patch_dir / 'metadata.json'
    if not meta.exists():
        return False
    data = json.loads(meta.read_text())
    return bool(data.get('train'))


def iter_train_images(
    patch_dir: Path,
    n_source_channels: int,
    channel_names: list[str],
    split: str = 'train',
):
    """Yield assembled (H, W, C) model inputs from saved patch source channels."""
    img_dir = patch_dir / split / 'images'
    meta = json.loads((patch_dir / 'metadata.json').read_text())
    for m in meta[split]:
        sid = m['sample_id']
        chans = []
        for i in range(n_source_channels):
            paths = [img_dir / f'{sid}_ch{i}.png']
            if i == 0 and n_source_channels == 1:
                paths.insert(0, img_dir / f'{sid}.png')
            arr = None
            for p in paths:
                arr = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if arr is not None:
                    break
            if arr is None:
                break
            chans.append(arr)
        if len(chans) != n_source_channels:
            continue
        yield build_image(chans, channel_names)

In [ ]:
#| export
def load_or_compute_stats(
    patch_dir: Path,
    n_source_channels: int,
    channel_names: list[str],
) -> tuple[tuple[float, ...], tuple[float, ...]]:
    """Load or compute per-channel mean/std from training patches."""
    stats_path = patch_dir / 'stats.json'
    if stats_path.exists():
        cached = json.loads(stats_path.read_text())
        mean = tuple(cached['mean'])
        std = tuple(cached['std'])
        print(f'Loaded stats from {stats_path}')
        return mean, std

    print('Computing per-channel mean/std from training patches …')
    mean, std = compute_dataset_stats(
        iter_train_images(patch_dir, n_source_channels, channel_names)
    )
    stats_path.write_text(json.dumps({'mean': list(mean), 'std': list(std)}, indent=2))
    print(f'Saved stats → {stats_path}')
    return mean, std


def load_or_compute_stats_direct(
    raw_dir: Path,
    train_ids: list[str],
    n_source_channels: int,
    channel_names: list[str],
    exts: str,
) -> tuple[tuple[float, ...], tuple[float, ...]]:
    """Load or compute per-channel mean/std from raw training images."""
    stats_path = raw_dir / 'stats.json'
    if stats_path.exists():
        cached = json.loads(stats_path.read_text())
        mean = tuple(cached['mean'])
        std = tuple(cached['std'])
        print(f'Loaded stats from {stats_path}')
        return mean, std

    print('Computing per-channel mean/std from raw training images …')
    mean, std = compute_dataset_stats(
        iter_raw_train_images(raw_dir, train_ids, n_source_channels, channel_names, exts)
    )
    stats_path.write_text(json.dumps({'mean': list(mean), 'std': list(std)}, indent=2))
    print(f'Saved stats → {stats_path}')
    return mean, std

In [ ]:
#| export
def run_patch_pipeline(cfg, raw_dir, patch_dir, n_source, channel_names, mask_suffix):
    """Build or reuse patch cache; return `(mean, std, patch_dir_str)`."""
    patch_cfg = PatchConfig(
        patch_size=cfg.patch.patch_size,
        stride=cfg.patch.stride,
        min_fg_pixels=cfg.patch.min_fg_pixels,
        max_empty_fraction=cfg.patch.max_empty_frac,
        val_fraction=cfg.patch.val_fraction,
        seed=cfg.patch.seed,
    )
    mask_cfg = resolve_mask_cfg(cfg)

    if cfg.data.force_rebuild or not patches_exist(patch_dir):
        print(f'\nBuilding patch dataset → {patch_dir}')
        patch_dir.mkdir(parents=True, exist_ok=True)
        meta = build_patch_dataset(
            source_dir=str(raw_dir),
            output_dir=str(patch_dir),
            cfg=patch_cfg,
            num_classes=cfg.train.num_classes,
            n_channels=n_source,
            n_jobs=cfg.patch.num_workers,
            fit_rects=cfg.data.fit_rects,
            image_exts=cfg.data.exts,
            mask_suffix=mask_suffix,
            **mask_cfg,
        )
        print(
            f'  train patches: {len(meta["train"])}  |  '
            f'val patches: {len(meta["val"])}'
        )
    else:
        print(f'\nReusing existing patches in {patch_dir} (set data.force_rebuild=true to rebuild)')

    mean, std = load_or_compute_stats(patch_dir, n_source, channel_names)
    return mean, std, str(patch_dir)


def run_direct_pipeline(cfg, raw_dir, sample_ids, n_source, channel_names, mask_suffix):
    """Split raw samples and compute stats; return `(mean, std, train_loader, val_loader)`."""
    train_ids, val_ids = split_sample_ids(
        sample_ids, cfg.patch.val_fraction, cfg.patch.seed,
    )
    print(f'\nDirect mode: {len(train_ids)} train / {len(val_ids)} val full images')
    mean, std = load_or_compute_stats_direct(
        raw_dir, train_ids, n_source, channel_names, cfg.data.exts,
    )
    boundary_blur = resolve_boundary_blur_cfg(cfg)
    mask_cfg = resolve_mask_cfg(cfg)
    train_loader, val_loader, _, _ = build_direct_dataloaders(
        raw_dir=str(raw_dir),
        train_ids=train_ids,
        val_ids=val_ids,
        batch_size=cfg.train.batch_size,
        num_workers=cfg.train.num_workers,
        crop_size=cfg.train.crop_size,
        num_classes=cfg.train.num_classes,
        n_source_channels=n_source,
        channel_names=channel_names,
        mask_suffix=mask_suffix,
        exts=cfg.data.exts,
        fit_rects=cfg.data.fit_rects,
        mean=mean,
        std=std,
        boundary_blur=boundary_blur,
        flips=bool(cfg.augment.get('flips', False)),
        seed=resolve_run_seed(cfg),
        **mask_cfg,
    )
    return mean, std, train_loader, val_loader


def validate_data_mode_channels(cfg: DictConfig) -> tuple[int, int, list[str]]:
    """Resolve channel preset and validate `model.in_channels` against `data_mode`."""
    preset = resolve_channel_preset(cfg.data.data_mode)
    n_source = preset['n_source_channels']
    channel_names = preset['channel_names']
    in_channels = int(cfg.model.in_channels)
    expected = preset['in_channels']
    if in_channels != expected:
        raise ValueError(
            f'model.in_channels={in_channels} does not match '
            f'data.data_mode={cfg.data.data_mode!r} '
            f'(expected {expected}). '
            f'Update model.in_channels in your config to match data_mode.'
        )
    return n_source, in_channels, channel_names


In [ ]:
#| export
def resolve_batch_aug_cfg(cfg: DictConfig) -> dict | None:
    """Map Hydra `augment.batch.*` to `train(batch_augs=...)`; None = disabled (strict no-op).

    Disabled when the master switch `augment.batch.enabled` is false **or** when every
    per-type probability is 0 — both must reproduce "no batch aug" training exactly.

    Example:
        >>> resolve_batch_aug_cfg(OmegaConf.create({'augment': {'batch': {'enabled': True, 'mixup': 0.05}}}))
        {'mosaic': 0.0, 'mixup': 0.05, 'cutmix': 0.0, 'copy_paste': 0.0, 'copy_paste_mode': 'flip'}
    """
    b = cfg.augment.get('batch', None)
    if b is None or not bool(b.get('enabled', False)):
        return None
    out = {
        'mosaic': float(b.get('mosaic', 0.0)),
        'mixup': float(b.get('mixup', 0.0)),
        'cutmix': float(b.get('cutmix', 0.0)),
        'copy_paste': float(b.get('copy_paste', 0.0)),
        'copy_paste_mode': str(b.get('copy_paste_mode', 'flip')),
    }
    if all(v == 0.0 for k, v in out.items() if k != 'copy_paste_mode'):
        return None
    return out

In [ ]:
# test: resolve_batch_aug_cfg — missing section / master off / all-zero -> None; enabled+probs -> dict
from omegaconf import OmegaConf

assert resolve_batch_aug_cfg(OmegaConf.create({'augment': {}})) is None
assert resolve_batch_aug_cfg(OmegaConf.create({'augment': {'batch': {'enabled': False, 'mixup': 0.05}}})) is None
assert resolve_batch_aug_cfg(OmegaConf.create(
    {'augment': {'batch': {'enabled': True, 'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0}}})) is None
out = resolve_batch_aug_cfg(OmegaConf.create(
    {'augment': {'batch': {'enabled': True, 'mixup': 0.05, 'copy_paste': 0.11, 'copy_paste_mode': 'flip'}}}))
assert out == {'mosaic': 0.0, 'mixup': 0.05, 'cutmix': 0.0, 'copy_paste': 0.11, 'copy_paste_mode': 'flip'}
print('resolve_batch_aug_cfg OK')

## Production inference knobs (`inference.*`)

Large-image inference is tiled (`predict_tiled`), and its knobs belong in config so a
production run needs no code change: `tile_size` (tile side fed to the model),
`overlap` (adjacent-tile overlap for Gaussian stitching), `inference_batch_size`
(tiles per forward — raise for throughput, lower if OOM; batch size never changes
results), and `prefetch_factor` for loader-driven tiled-inference consumers
(`predict_tiled` batches in-process and ignores it).

Experiment first — hand-roll the mapping and its validation:

In [ ]:
# experiment: map inference.* by hand; see which bad values must fail loudly
from omegaconf import OmegaConf

cfg = OmegaConf.create({'inference': {'tile_size': 512, 'overlap': 64}})
inf = cfg.get('inference', None) or {}
tile_size, overlap = int(inf.get('tile_size', 256)), int(inf.get('overlap', 32))
bs = int(inf.get('inference_batch_size', 8))
pf = inf.get('prefetch_factor', None)
print('defaults fill in:', {'tile_size': tile_size, 'overlap': overlap,
                             'inference_batch_size': bs, 'prefetch_factor': pf})
assert 0 <= overlap < tile_size and bs >= 1 and (pf is None or pf >= 1)
# bad values: overlap >= tile_size, batch < 1, prefetch < 1 — each must violate a rule
full = OmegaConf.create({'inference': {'tile_size': 512, 'overlap': 64, 'inference_batch_size': 4, 'prefetch_factor': 3}})
for bad in ({'overlap': 512}, {'inference_batch_size': 0}, {'prefetch_factor': 0}):
    i2 = OmegaConf.merge(full, {'inference': bad}).inference
    violated = (not (0 <= int(i2.overlap) < int(i2.tile_size))) or int(i2.inference_batch_size) < 1 \
        or (i2.prefetch_factor is not None and int(i2.prefetch_factor) < 1)
    assert violated, f'bad config slipped through: {bad}'
    print('would reject:', bad)
print('validation rules pinned')

In [ ]:
#| export
def resolve_inference_cfg(cfg: DictConfig) -> dict:
    """Map Hydra `inference.*` to the production tiled-inference knobs of `predict_tiled`.

    Missing block -> defaults (`tile_size=256`, `overlap=32`, `inference_batch_size=8`,
    `prefetch_factor=None`). Values are validated so bad config fails before the GPU is
    touched. `prefetch_factor` targets loader-driven tiled-inference consumers;
    `predict_tiled` batches tiles in-process and ignores it.

    Example:
        >>> resolve_inference_cfg(OmegaConf.create({'inference': {'tile_size': 512, 'overlap': 64}}))
        {'tile_size': 512, 'overlap': 64, 'inference_batch_size': 8, 'prefetch_factor': None}
    """
    inf = cfg.get('inference', None) or {}
    tile_size = int(inf.get('tile_size', 256))
    overlap = int(inf.get('overlap', 32))
    inference_batch_size = int(inf.get('inference_batch_size', 8))
    prefetch = inf.get('prefetch_factor', None)
    prefetch_factor = int(prefetch) if prefetch is not None else None
    if tile_size < 1:
        raise ValueError(f'inference.tile_size must be >= 1, got {tile_size}')
    if not (0 <= overlap < tile_size):
        raise ValueError(f'inference.overlap must satisfy 0 <= overlap < tile_size ({tile_size}), got {overlap}')
    if inference_batch_size < 1:
        raise ValueError(f'inference.inference_batch_size must be >= 1, got {inference_batch_size}')
    if prefetch_factor is not None and prefetch_factor < 1:
        raise ValueError(f'inference.prefetch_factor must be >= 1 or null, got {prefetch_factor}')
    return {'tile_size': tile_size, 'overlap': overlap,
            'inference_batch_size': inference_batch_size, 'prefetch_factor': prefetch_factor}

In [ ]:
#| test
# test: inference resolver — defaults, explicit values, loud failures
from omegaconf import OmegaConf

assert resolve_inference_cfg(OmegaConf.create({})) == \
    {'tile_size': 256, 'overlap': 32, 'inference_batch_size': 8, 'prefetch_factor': None}
assert resolve_inference_cfg(OmegaConf.create({'inference': {
    'tile_size': 512, 'overlap': 64, 'inference_batch_size': 4, 'prefetch_factor': 3}})) == \
    {'tile_size': 512, 'overlap': 64, 'inference_batch_size': 4, 'prefetch_factor': 3}
assert resolve_inference_cfg(OmegaConf.create({'inference': {'prefetch_factor': None}}))['prefetch_factor'] is None
for bad in ({'overlap': 512}, {'inference_batch_size': 0}, {'prefetch_factor': 0}, {'tile_size': 0}):
    try:
        resolve_inference_cfg(OmegaConf.create({'inference': bad}))
        raise AssertionError(f'no error for {bad}')
    except ValueError:
        pass

## Hierarchy-loss weight (`train.hierarchy.weight`, #67)

`train.hierarchy.<sub>_subset_of_<super>: true` already names *which* classes must nest
(`resolve_hierarchy_pairs`). Nothing said *how hard* training should push, so the loss stayed off.
One new key in the same block: `train.hierarchy.weight` (float, default `0.0` = off).
It is the run-wide fallback; a loss phase may still set its own `w_hierarchy`.
Missing section, missing key or `null` all mean `0.0`; a negative or non-numeric value must fail loudly.

In [ ]:
# experiment: read the weight by hand and see what bad values look like
from omegaconf import OmegaConf
for h in [None, {}, {'lifted_subset_of_chip': True}, {'lifted_subset_of_chip': True, 'weight': 0.5}, {'weight': None}, {'weight': -1}, {'weight': 'x'}]:
    c = OmegaConf.create({'train': {'hierarchy': h}})
    w = (c.train.get('hierarchy') or {}).get('weight')
    print(repr(h), '->', w)

In [ ]:
#| export
def resolve_hierarchy_weight(cfg: DictConfig) -> float:
    """Read `train.hierarchy.weight` (hierarchy-loss weight); `0.0` (off) when the section or key is absent.

    Raises `ValueError` for negative or non-numeric values. Pairs come from `resolve_hierarchy_pairs`.
    """
    raw = (cfg.train.get('hierarchy', None) or {}).get('weight', None)
    if raw is None:
        return 0.0
    try:
        weight = float(raw)
    except (TypeError, ValueError):
        raise ValueError(f'train.hierarchy.weight must be a number >= 0, got {raw!r}') from None
    if weight < 0:
        raise ValueError(f'train.hierarchy.weight must be >= 0, got {weight}')
    return weight

In [ ]:
#| test
# test: resolve_hierarchy_weight — absent = 0.0 (off); value read; bad values fail loudly; weight key is not mistaken for a pair
from omegaconf import OmegaConf
mk = lambda h: OmegaConf.create({'train': {'hierarchy': h}})
assert resolve_hierarchy_weight(OmegaConf.create({'train': {}})) == 0.0
assert resolve_hierarchy_weight(mk(None)) == 0.0 and resolve_hierarchy_weight(mk({})) == 0.0
assert resolve_hierarchy_weight(mk({'lifted_subset_of_chip': True})) == 0.0
assert resolve_hierarchy_weight(mk({'weight': None})) == 0.0
assert resolve_hierarchy_weight(mk({'lifted_subset_of_chip': True, 'weight': 0.5})) == 0.5
assert resolve_hierarchy_pairs(mk({'lifted_subset_of_chip': True, 'weight': 0.5}), ['chip', 'void', 'lifted']) == [(2, 0)]
for bad in (-1, 'x'):
    try: resolve_hierarchy_weight(mk({'weight': bad}))
    except ValueError: pass
    else: raise AssertionError(f'expected ValueError for weight={bad!r}')
print('resolve_hierarchy_weight ✓  default off, value read, negative/non-numeric fail, coexists with pair keys')

## Training schedule (`model.source: custom_unet`, #61)

`smp` encoders start from ImageNet weights, so `train()` freezes them for a few epochs and gives them a lower LR.
A from-scratch UNet would be starved by that, so `custom_unet` gets `freeze_epochs=0` and `encoder_lr=decoder_lr`
unless the user sets them explicitly under `model.custom_unet`.

In [ ]:
from omegaconf import OmegaConf
# experiment: the rule, inline on synthetic cfgs
_t = OmegaConf.create({'model': {'source': 'custom_unet'}, 'train': {'freeze_epochs': 5, 'encoder_lr': 1e-4, 'decoder_lr': 1e-3}})
def _rule(c):
    t, cu = c.train, c.model.get('custom_unet', None) or {}
    if c.model.get('source', 'smp') != 'custom_unet': return t.freeze_epochs, t.encoder_lr, t.decoder_lr
    fe, el = cu.get('freeze_epochs', None), cu.get('encoder_lr', None)
    return (0 if fe is None else fe), (t.decoder_lr if el is None else el), t.decoder_lr
print(_rule(_t), _rule(OmegaConf.merge(_t, {'model': {'custom_unet': {'freeze_epochs': 2}}})), _rule(OmegaConf.merge(_t, {'model': {'source': 'smp'}})))

In [ ]:
# verify: scratch default, explicit override, smp passthrough
assert _rule(_t) == (0, 1e-3, 1e-3)
assert _rule(OmegaConf.merge(_t, {'model': {'custom_unet': {'freeze_epochs': 2}}})) == (2, 1e-3, 1e-3)
assert _rule(OmegaConf.merge(_t, {'model': {'source': 'smp'}})) == (5, 1e-4, 1e-3)

In [ ]:
#| export
from typing import NamedTuple
import warnings

from segmentation_tools_pt.model import _custom_unet_cfg

class TrainSchedule(NamedTuple):
    "Encoder freeze length and learning rates handed to `train()`; unpacks like a plain tuple."
    freeze_epochs: int
    encoder_lr: float
    decoder_lr: float

def resolve_train_schedule(
    cfg,   # Hydra config with `model.*` and `train.{freeze_epochs,encoder_lr,decoder_lr}`
) -> TrainSchedule:
    """Return `TrainSchedule(freeze_epochs, encoder_lr, decoder_lr)` for `train()`, neutralising pretrained-encoder settings for `custom_unet`.

    `custom_unet` is random-init, so a frozen warmup and a lower encoder LR would starve it. Unless
    `model.custom_unet.freeze_epochs` / `encoder_lr` are set explicitly, it gets `freeze_epochs=0` and
    `encoder_lr=decoder_lr`, with a warning. `smp` values pass through unchanged.

    Example:
        >>> resolve_train_schedule(cfg)  # doctest: +SKIP
    """
    train_cfg = cfg.train
    freeze, enc_lr, dec_lr = train_cfg.freeze_epochs, train_cfg.encoder_lr, train_cfg.decoder_lr
    if cfg.model.get('source', 'smp') != 'custom_unet':
        return TrainSchedule(freeze, enc_lr, dec_lr)
    custom_cfg = _custom_unet_cfg(cfg.model)
    freeze_out = custom_cfg.get('freeze_epochs', None)
    encoder_lr_out = custom_cfg.get('encoder_lr', None)
    freeze_out = 0 if freeze_out is None else freeze_out
    encoder_lr_out = dec_lr if encoder_lr_out is None else encoder_lr_out
    if (freeze_out, encoder_lr_out) != (freeze, enc_lr):
        warnings.warn(
            f'model.source=custom_unet is random-init: using freeze_epochs={freeze_out} and encoder_lr={encoder_lr_out} '
            f'instead of train.freeze_epochs={freeze} / train.encoder_lr={enc_lr}. '
            'Set model.custom_unet.freeze_epochs / model.custom_unet.encoder_lr to override.', UserWarning)
    return TrainSchedule(freeze_out, encoder_lr_out, dec_lr)

In [ ]:
#| test
# test: custom_unet is random-init, so the pretrained-encoder schedule (frozen warmup, 10x lower encoder LR) is neutralised
sched_cfg = OmegaConf.create({'model': {'source': 'custom_unet'}, 'train': {'freeze_epochs': 5, 'encoder_lr': 1e-4, 'decoder_lr': 1e-3}})
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    assert resolve_train_schedule(sched_cfg) == (0, 1e-3, 1e-3)
assert any('custom_unet' in str(x.message) and 'freeze_epochs' in str(x.message) for x in w)
# explicit override under model.custom_unet wins
ov = OmegaConf.merge(sched_cfg, {'model': {'custom_unet': {'freeze_epochs': 2, 'encoder_lr': 5e-4}}})
assert resolve_train_schedule(ov) == (2, 5e-4, 1e-3)
# smp is untouched
smp_s = OmegaConf.merge(sched_cfg, {'model': {'source': 'smp'}})
assert resolve_train_schedule(smp_s) == (5, 1e-4, 1e-3)
assert resolve_train_schedule(OmegaConf.create({'model': {}, 'train': {'freeze_epochs': -1, 'encoder_lr': 1e-4, 'decoder_lr': 1e-3}})) == (-1, 1e-4, 1e-3)
# named fields, still unpacks like a tuple
sch = resolve_train_schedule(ov)
assert (sch.freeze_epochs, sch.encoder_lr, sch.decoder_lr) == (2, 5e-4, 1e-3) and sch._fields == ('freeze_epochs', 'encoder_lr', 'decoder_lr')
fz, elr, dlr = sch
assert (fz, elr, dlr) == tuple(sch)
